In [36]:
# Chromeブラウザを利用して、スクレイピングをするため、Chromeブラウザをインストール
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Hit:1 http://dl.google.com/linux/chrome/deb stable InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:4 https://dl.google.com/linux/chrome-stable/deb stable InRelease [1,825 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://dl.google.com/linux/chrome-stable/deb stable/main amd64 Packages [1,219 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Fetched 3,044 B in 2s (1,921 B/s)
Reading package lists... Done
W: http://dl.google.com/linux/chrome/deb/di

In [37]:
# SeleniumとChromeDriverを自動で管理するためのインストール
!pip install selenium
!pip install webdriver-manager

In [38]:
!pip install beautifulsoup4

In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import pandas as pd
import time

In [13]:
#  Chromeブラウザ起動オプションを指定
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

In [14]:
# ChromeDriverManagerを使用してChromeDriverを自動的に管理
service = Service(ChromeDriverManager().install())

# インスタンスを作成し、変数chrome_driverに格納
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)

In [15]:
# 日経新聞のWebサイトを開く
chrome_driver.get('https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month')

In [43]:
#↑ ここまで成功...(?)

In [16]:
# HTML解析（BeautifulSoup）
def extract_stock_data(html):
  soup = BeautifulSoup(html, 'html.parser')
  try:
    date = soup.select_one(".highcharts-tooltip td:nth-child(1)").text.strip()
    open_price = soup.select_one(".highcharts-tooltip td:nth-child(2)").text.strip()
    high_price = soup.select_one(".highcharts-tooltip td:nth-child(3)").text.strip()
    low_price = soup.select_one(".highcharts-tooltip td:nth-child(4)").text.strip()
    close_price = soup.select_one(".highcharts-tooltip td:nth-child(5)").text.strip()

    return [date, open_price, high_price, low_price, close_price]
  except:
    return None

In [17]:






def get_stock_values(driver, url):
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.action_chains import ActionChains
    import time
    from datetime import datetime, timedelta

    driver.get(url)
    time.sleep(3)

    actions = ActionChains(driver)

    graph = driver.find_element(By.CLASS_NAME, "highcharts-container")
    width = graph.size["width"]

    actions.move_to_element(graph).perform()

    results = []
    seen_dates = set()  # ←重複防止用

    # 今日から1ヶ月前の日付
    one_month_ago = datetime.today() - timedelta(days=30)

    for i in range(width):
        x_offset = width // 2 - i
        actions.move_to_element_with_offset(graph, x_offset, 0).perform()
        time.sleep(0.01)

        html = driver.page_source
        data = extract_stock_data(html)

        if data:
            date_str = data[0]  # 日付

            # ▼ 日付フォーマット調整（例：2026/04/22 → 2026-04-22）
            date_str = date_str.replace("/", "-")

            try:
                date_obj = datetime.strptime(date_str, "%Y-%m-%d")
            except:
                continue

            # ✔ ① 1ヶ月より前なら終了
            if date_obj < one_month_ago:
                break

            # ✔ ② 重複除外
            if date_str not in seen_dates:
                seen_dates.add(date_str)
                results.append(data)

    return results
# メイン処理
if __name__ == '__main__':
  # ドライバ準備
  service = Service(ChromeDriverManager().install())
  # Chromeオプションを適用してドライバーを作成
  driver = webdriver.Chrome(service=service, options=chrome_options)

  url = 'https://www.nikkei.com/markets/worldidx/chart/nk225/?type=6month'

  # 開始時間
  start_time = time.time()

  # データ取得
  stock_data_list = get_stock_values(driver,url)

  # 終了時間
  end_time = time.time()

  # 実行時間
  print(f'スクレイピング時間：{end_time - start_time:.2f}秒')

  # 結果表示
  for data in stock_data_list:
    print(f'日付：{data[0]}, 始値：{data[1]}, 高値：{data[2]}, 安値：{data[3]}, 終値：{data[4]}')

  driver,quit()

スクレイピング時間：129.03秒
日付：2026/4/22, 始値：始値: 59,104.11, 高値：高値: 59,708.21, 安値：安値: 59,005.48, 終値：終値: 59,574.35
日付：2026/4/21, 始値：始値: 59,031.51, 高値：高値: 59,611.91, 安値：安値: 59,004.76, 終値：終値: 59,349.17
日付：2026/4/20, 始値：始値: 58,821.16, 高値：高値: 59,169.13, 安値：安値: 58,687.96, 終値：終値: 58,824.89
日付：2026/4/17, 始値：始値: 59,255.09, 高値：高値: 59,381.25, 安値：安値: 58,475.9, 終値：終値: 58,475.9
日付：2026/4/16, 始値：始値: 58,479.83, 高値：高値: 59,688.1, 安値：安値: 58,428.19, 終値：終値: 59,518.34
日付：2026/4/15, 始値：始値: 58,265.18, 高値：高値: 58,585.95, 安値：安値: 58,028.75, 終値：終値: 58,134.24
日付：2026/4/14, 始値：始値: 57,085.65, 高値：高値: 57,979.82, 安値：安値: 57,010.18, 終値：終値: 57,877.39
日付：2026/4/13, 始値：始値: 56,421.46, 高値：高値: 56,765.72, 安値：安値: 56,232.78, 終値：終値: 56,502.77
日付：2026/4/10, 始値：始値: 56,265.77, 高値：高値: 57,012.77, 安値：安値: 56,251.18, 終値：終値: 56,924.11
日付：2026/4/9, 始値：始値: 56,199.86, 高値：高値: 56,406.49, 安値：安値: 55,763.05, 終値：終値: 55,895.32
日付：2026/4/8, 始値：始値: 54,386.65, 高値：高値: 56,424.63, 安値：安値: 54,380.02, 終値：終値: 56,308.42
日付：2026/4/7, 始値：始値: 53,571.28, 高値：高値: 53,916.35, 安値：